# 2D toric code — hz sweep convergence

Training curves for the known analytic cut **hx=hy=0, sweep hz** (2nd-order,
3D-Ising, $h_z^c\approx0.328\,J$). Combo CNN, OBC, per-hz NQS runs.

Per system size $L$ (two panels): **energy vs step** (dashed line = the
$h=0$ anchor $E_0=-(L^2+(L-1)^2)$; a converged finite-field run must sit
*below* it) and **V-score vs step** ($=N\,\mathrm{Var}/E^2$, log scale —
the convergence gauge). Curves colored by $h_z$.

Source: `{name}.curve.json` files pulled from
`$PSCRATCH/tc_nqs_2d/phase_hx0.0/L*/` into `results/tc2d_hx0.0_curves/`.
MCMC acceptance was not logged to the curve, so it is omitted.

In [ ]:
import glob, json, os, re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm, colors

CURVE_DIR = 'results/tc2d_hx0.0_curves'

def anchor_E0(L):
    """Unperturbed OBC ground-state energy = -(#A_v + #B_p) = -(L^2 + (L-1)^2)."""
    return -(L**2 + (L - 1)**2)

def load_curves(curve_dir=CURVE_DIR):
    """{L: [ (hz, curve_dict, completed_steps), ... sorted by hz ]}."""
    out = {}
    for p in sorted(glob.glob(os.path.join(curve_dir, '*.curve.json'))):
        d = json.load(open(p))
        cfg = d.get('config', {})
        L = int(cfg['L']) if 'L' in cfg else int(re.search(r'_L(\d+)_', p).group(1))
        hz = float(cfg['hz']) if 'hz' in cfg else float(re.search(r'hz([0-9.]+)_', p).group(1))
        out.setdefault(L, []).append((hz, d['curve'], d.get('completed_steps')))
    for L in out:
        out[L].sort(key=lambda t: t[0])
    return out

curves = load_curves()
print('loaded L =', sorted(curves))
for L in sorted(curves):
    print(f'  L={L}: {len(curves[L])} hz points, '
          f"steps {min(c[2] for c in curves[L])}..{max(c[2] for c in curves[L])}")

In [ ]:
def plot_L(L, entries):
    hz_vals = np.array([e[0] for e in entries])
    norm = colors.Normalize(vmin=hz_vals.min(), vmax=hz_vals.max())
    cmap = cm.viridis

    fig, (ax_e, ax_v) = plt.subplots(1, 2, figsize=(13, 4.6))
    for hz, cv, _ in entries:
        c = cmap(norm(hz))
        step = np.asarray(cv['step'])
        ax_e.plot(step, cv['energy'], color=c, lw=1.2)
        ax_v.semilogy(step, np.abs(cv['vscore']), color=c, lw=1.2)

    a = anchor_E0(L)
    ax_e.axhline(a, ls='--', color='crimson', lw=1.2, zorder=0)
    ax_e.text(0.98, 0.96, f'$h{{=}}0$ anchor $E_0={a}$', color='crimson',
              ha='right', va='top', transform=ax_e.transAxes, fontsize=9)
    ax_e.set(xlabel='training step', ylabel='energy $E$',
             title=f'$L={L}$  ($N={2*L*L-2*L}$) \u2014 energy')
    ax_v.set(xlabel='training step', ylabel='V-score  $N\\,\\mathrm{Var}/E^2$',
             title=f'$L={L}$ \u2014 V-score')
    ax_v.grid(True, which='both', alpha=0.25)
    ax_e.grid(True, alpha=0.25)

    sm = cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cb = fig.colorbar(sm, ax=[ax_e, ax_v], pad=0.02, fraction=0.04)
    cb.set_label('$h_z$')
    cb.ax.axhline(0.328, color='crimson', lw=1.5)   # 3D-Ising hz_c
    return fig

for L in sorted(curves):
    plot_L(L, curves[L])
plt.show()

---
# FM order parameter → finite-size extrapolation

Consumes the tiny per-L JSONs written on the cluster by
`python -m analysis.fm_2d ... --out fm2d_L{L}_{sector}.json` and pulled into
`results/tc2d_{electric,magnetic}/`. Pure numpy/scipy — no NetKet.

For each sector we fit a logistic $a+b/(1+e^{-(h-h_0)/w})$ to $O_{FM}(h)$; its
inflection $h_0$ is the pseudo-critical $h_c(L)$. A finite-difference derivative
peak is the model-free cross-check. Then $h_c(L)$ vs $1/L$ extrapolates to
$L\to\infty$, compared against the analytic **3D-Ising $h_c\approx0.328\,J$**
(same value for the electric $h_z$ cut and, by e–m self-duality, the magnetic
$h_x$ cut).

In [ ]:
import glob, json, os
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

H_C_ANALYTIC = 0.328   # 3D-Ising / (2+1)D TFIM critical field (Vidal-Dusuel-Schmidt 2009)

def logistic(h, a, b, h0, w):
    return a + b / (1.0 + np.exp(-(h - h0) / w))

def dlogistic(h, a, b, h0, w):
    z = np.exp(-(h - h0) / w)
    return (b / w) * z / (1.0 + z) ** 2

def fit_sigmoid(x, y, ye=None):
    x, y = np.asarray(x, float), np.asarray(y, float)
    p0 = [y[0], y[-1] - y[0], float(np.median(x)), 0.1 * (x[-1] - x[0]) or 0.1]
    kw = dict(sigma=np.asarray(ye, float), absolute_sigma=True) \
         if (ye is not None and np.all(np.asarray(ye) > 0)) else {}
    try:
        popt, pcov = curve_fit(logistic, x, y, p0=p0, maxfev=20000, **kw)
        return popt, float(np.sqrt(abs(pcov[2, 2])))
    except Exception as exc:
        print('  [fit] logistic failed:', exc); return None, float('nan')

def load_sector(directory):
    """Sorted list of per-L FM records for one sector dir (fm2d_L*_*.json)."""
    recs = [json.load(open(jp)) for jp in sorted(glob.glob(os.path.join(directory, 'fm2d_L*.json')))]
    return sorted(recs, key=lambda r: r['L'])

SECTOR_DIRS = {'electric': 'results/tc2d_electric', 'magnetic': 'results/tc2d_magnetic'}
sectors = {s: load_sector(d) for s, d in SECTOR_DIRS.items() if os.path.isdir(d) and load_sector(d)}
print('sectors with data:', {s: [r['L'] for r in recs] for s, recs in sectors.items()} or 'NONE yet')

In [ ]:
def analyze_sector(sector, recs, vscore_max=1e-2):
    """Per-L sigmoid fit + FSS for one sector. Returns (Ls, hc, hc_err). Draws 3 panels."""
    field_name = recs[0].get('field_name', 'hz' if sector == 'electric' else 'hx')
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
    colors = plt.cm.viridis(np.linspace(0, 0.82, len(recs)))
    Ls, hcs, hces = [], [], []
    for rec, c in zip(recs, colors):
        L = rec['L']
        h  = np.array(rec['field']); O = np.array(rec['O']); Oe = np.array(rec['Oe'])
        vs = np.array(rec.get('vscore', np.full_like(h, np.nan)))
        keep = ~(np.isfinite(vs) & (vs > vscore_max))   # drop unconverged points from the fit
        if (~keep).any():
            print(f'  {sector} L={L}: dropped {int((~keep).sum())} pt(s) with Vscore>{vscore_max:g}')
        h, O, Oe = h[keep], O[keep], Oe[keep]
        p, e = fit_sigmoid(h, O, Oe)
        hh = np.linspace(h.min(), h.max(), 400)
        hc = p[2] if p is not None else np.nan
        lbl = f'L={L}: $h_c$={hc:.3f}' + (f'±{e:.3f}' if np.isfinite(e) else '')
        ax[0].errorbar(h, O, yerr=Oe, fmt='o', ms=4, capsize=2, color=c, label=lbl)
        if p is not None:
            ax[0].plot(hh, logistic(hh, *p), '-', color=c)
            ax[0].axvline(hc, ls='--', color=c, lw=0.8, alpha=0.5)
            ax[1].plot(hh, dlogistic(hh, *p), '-', color=c, label=f'L={L}, max={hc:.3f}')
            Ls.append(L); hcs.append(hc); hces.append(e)
        ax[1].plot(h, np.gradient(O, h), 'o', ms=3, color=c, alpha=0.5)
    ax[0].axvline(H_C_ANALYTIC, color='crimson', lw=1.4, ls=':', label=f'analytic {H_C_ANALYTIC}')
    ax[0].set(xlabel=f'${field_name}$', ylabel='$O_{FM}$', title=f'{sector}: FM order parameter')
    ax[0].legend(fontsize=8)
    ax[1].axvline(H_C_ANALYTIC, color='crimson', lw=1.4, ls=':')
    ax[1].set(xlabel=f'${field_name}$', ylabel=f'$dO_{{FM}}/d{field_name}$',
              title='derivative — peak = transition'); ax[1].legend(fontsize=8)
    # FSS: h_c(L) vs 1/L, linear L->inf
    if len(Ls) >= 2:
        Ls_a = np.array(Ls, float); hcs_a = np.array(hcs); hces_a = np.array(hces)
        x = 1.0 / Ls_a; m, b = np.polyfit(x, hcs_a, 1)
        xs = np.linspace(0, x.max() * 1.05, 50)
        ax[2].errorbar(x, hcs_a, yerr=hces_a, fmt='o', capsize=3, color='C0')
        ax[2].plot(xs, m * xs + b, '-', color='C0', lw=1.2)
        ax[2].axhline(b, ls=':', color='k', label=f'$h_c(\\infty)$={b:.3f}')
        ax[2].axhline(H_C_ANALYTIC, color='crimson', lw=1.4, ls=':', label=f'analytic {H_C_ANALYTIC}')
        ax[2].set(xlabel='$1/L$', ylabel='$h_c(L)$', title='finite-size scaling')
        ax[2].legend(fontsize=8)
        print(f'[{sector}] h_c(inf)={b:.4f}  (analytic {H_C_ANALYTIC}; '
              f'diff {b - H_C_ANALYTIC:+.4f}); per-L h_c={dict(zip(Ls, np.round(hcs,4)))}')
    else:
        ax[2].axis('off')
    fig.suptitle(f'2D toric code — {sector} sector ({field_name} sweep)')
    fig.tight_layout()
    return (np.array(Ls), np.array(hcs), np.array(hces))

fss = {s: analyze_sector(s, recs) for s, recs in sectors.items()}
plt.show()